# Time resolved UV-VIS

In this example we will combine what we have learned in the previous examples to study time evolution of UV-VIS spectra. Time-resolved measurements are widely used to study reaction kinetics.

In [ ]:
#### as usual we need to run various imports
import numpy as np
import scipy as sp
from matplotlib import pyplot as plt

from scipy.signal import find_peaks
from scipy import ndimage

%matplotlib inline


The time-resolved experiment we are considering here is the extraction of polyphenols from camellia sinensis. It is a series of over 800 UV vis spectra, with one measurement taken per second. This is an experiment many of you have probably already performed, but here we are able to obtain time-resolved UV-VIS spectra of the process. First you should make sure the raw data folder is in the same directory as the notebook, so the following cell can open correctly:

In [ ]:
#this is a single UV-vis trace
data=np.loadtxt('data/runs/run_1.txt')

You should have the folders "runs" and "blank_cell" within the data folder. Assuming we were able to load the data correctly, if we plot it we can see the raw intensity (in counts) as a function of wavelength.

In [ ]:
plt.plot(data[:,0],data[:,1])
plt.ylabel('Intensity / Counts', fontsize=16)
plt.xlabel('Wavelength / nm', fontsize=16)

Normally when we consider UV-Vis data we might want to know the absorbance or transmittance, so we need to compare this data to a blank (a cell containing just solvent). We also note that the first few data points and last few data points are very noisy so in some cases we end up with negative absorbance! We can exclude them from our analysis via: data=data[the_number_of_points_at_the_start_to_exclude:-the_number_of_points_at_the_end_to_exclude]. Try changing these numbers for both the blank and the data, but make sure the numbers are the same for both - if they are different we get an error because later we divide one array by the other, which only works if they are of the same length - see for yourself. To prevent this from happening we could define a start and end for both, such that: data=data[start:-end] for both, and define start and end separately.

In [ ]:

plt.ylabel('Absorbance', fontsize=16)
plt.xlabel('Wavelength / nm', fontsize=16)
#here we are loading our data and the blank. Note that for very high and low wavelength we have more-or-less no signal
#we can exclude these very noisy data points with data=data[80:-300] - these excludes the first 80 and last 300 points

datasample=np.loadtxt('data/runs/run_100.txt')
datasample=datasample[0:-1]
datablank=np.loadtxt('data/blank_cell/blank.txt')
datablank=datablank[0:-1]
#here we are simply converting to absorbance by taking the log of the data divided by the blank - recall analytical from part 2!
absorbance=-np.log10(datasample[:,1]/datablank[:,1])
plt.plot(datasample[0:-1,0],absorbance[0:-1])

We can now see how the absorbance changes with time. In the following cell we are using a loop, as we saw in class. Now we are using the loop to repeatedly plot UV-Vis data. In the line where we call the loop we are iterating f between 0 and 834 (the total number of UV-Vis files we have). Rather than going one by one we are going 50 at a time, hence for f in range(start,end,interval). We are then using the current f value to feed into the file name, loading the data, working out the absorbance, plotting it, then repeating for the next f value. The reason we have an interval of 50 is quite simple - the plot gets very busy if we plot everything, so it is best to subsample. Try for yourself by changing the interval to a smaller number.

In [ ]:
datablank=np.loadtxt('data/blank_cell/blank.txt')
datablank=datablank[80:-300]

plt.ylabel('Absorbance', fontsize=16)
plt.xlabel('Wavelength / nm', fontsize=16)
#this is our loop, which increases f from 0, to 50, 100 and so on.
for f in range(0,834,50):
    #here we are working out the absorbance and plotting it.
    datasample=np.loadtxt('data/runs/run_'+str(f)+'.txt')
    datasample=datasample[80:-300]
    absorbance=-np.log10(datasample[:,1]/datablank[:,1])
    plt.plot(datasample[0:-1,0],absorbance[0:-1])


# Finding Absorbance

We are interested in determining the time evolution of the absorbance from this data. This data has a single main peak to fit, so we have a few options to fit it: We could determine the index (i.e., the number in the list of wavelengths) that corresponds to the maximum of absorbance for each run. We could also simply select a wavelength/index, and see how the absorbance changes with time for that wavelength. Because our absorbance is very low at early times it can be quite difficult to fit a peak, so the simplest option is to determine the index of the maximum of absorbance. This is also a computationally cheap task compared to fitting a function to the data, so it is quite fast - this is useful when we want to run this 834 times!

In [ ]:
#Here we are loading our file and determining absorbance as above.
datablank=np.loadtxt('data/blank_cell/blank.txt')
datablank=datablank[80:-300]

f=833
datasample=np.loadtxt('data/runs/run_'+str(f)+'.txt')
datasample=datasample[80:-300]
absorbance=-np.log10(datasample[:,1]/datablank[:,1])

#here we are determining the index (the position in the list of wavelengths) corresponding to the largest value of absorbance
#using the np.argmax function
index_max = np.argmax(absorbance[10:-1])
print(index_max)

We can therefore do this for all of our data sets, determining the absorbance for every UV-Vis dataset in a few seconds.

1: Produce a script that automatically finds the peak absorbance for the above index_max for the entire dataset, recording these values to a list or array. Plot the absorbance as a function of time.

2: Improve the above script by finding the index_max for each run, recording both the peak position and the peak absorbance. Plot both the absorbance and peak position as a function of time, and also plot peak absorbance versus peak position.

3: Choose a figure of merit to determine when the extraction is 'finished' - we will call this t_f. By using an image filter to smooth the data, produce a plot of t_f as a function of filter width. Based on the plot, provide an estimate (with error margin) of how long the extraction should proceed for. You may find it easier to 'functionalise' each step for easier to read code.

Advanced task: If you found all the previous tasks relatively easy and there is still plenty of time left, try and do this same analysis again - but this time determine the absorbance by fitting the peak shape rather than by taking the maximum absorbance. Produce a figure of absorbance versus time as before, and a figure of peak wavelength versus time. Then as before, produce a plot of t_f as a function of filter width. In principle this approach should be more accurate because the maximum value of absorbance could correspond to noise rather than the peak of the function, whereas fitting the entire peak shape is less sensitive to noise.